# CS5785 Midterm Project 2026

This notebook includes baseline models and code for preparing the two Kaggle submissions. See the project PDF for the full requirements.

## Load the data

Download the CSV files from both competitions and place them in the same folder as this notebook.

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, log_loss

In [ ]:
reason_train = pd.read_csv("part1_train.csv")
reason_val = pd.read_csv("part1_val.csv")
reason_test = pd.read_csv("part1_test.csv")

outcome_train = pd.read_csv("outcome_train.csv")
outcome_val = pd.read_csv("outcome_val.csv")
outcome_test = pd.read_csv("outcome_test.csv")

pathway_catalog = pd.read_csv("pathway_catalog.csv")

## Part 1: Stopping reasons

Each trial can have several reason labels. In the training set, `-100` marks missing labels.

### Prepare the data

Use the 150 embedding columns and standardize them using the training data.

In [ ]:
labels = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
features = [f"emb_{i}" for i in range(150)]
labeled = (reason_train[labels] != -100).all(axis=1)

scaler = StandardScaler()
X_train = scaler.fit_transform(reason_train[features])
X_val = scaler.transform(reason_val[features])
X_test = scaler.transform(reason_test[features])

y_train = reason_train.loc[labeled, labels]
y_val = reason_val[labels]

print("Labeled trials:", labeled.sum())
print("Unlabeled trials:", (~labeled).sum())

### Logistic regression baseline

Train on the labeled trials and evaluate on the validation set.

In [ ]:
reason_model = OneVsRestClassifier(
    LogisticRegression(max_iter=500, solver="liblinear", random_state=42)
)
reason_model.fit(X_train[labeled], y_train)
val_pred = reason_model.predict(X_val)

print("Sample-wise F1:", round(f1_score(y_val, val_pred, average="samples", zero_division=0), 4))
print("Macro F1:", round(f1_score(y_val, val_pred, average="macro", zero_division=0), 4))

per_label_f1 = f1_score(y_val, val_pred, average=None, zero_division=0)
pd.Series(per_label_f1, index=labels, name="F1").round(4)

### Your method

Use at least one unsupervised method, such as K-means or GMM, to recover the missing training labels. Evaluate your method on the validation set.

In [ ]:
# Add your label-recovery method here.

### Save Part 1 predictions

Save test predictions as `ID,Predicted`, with reason names separated by spaces. This example uses the baseline model.

In [ ]:
test_pred = reason_model.predict(X_test)
predicted_reasons = []

for row in test_pred:
    names = [label for label, value in zip(labels, row) if value == 1]
    predicted_reasons.append(" ".join(names))

part1_submission = reason_test[["ID"]].copy()
part1_submission["Predicted"] = predicted_reasons
part1_submission.to_csv("part1_submission.csv", index=False)
part1_submission.head()

## Part 2: Trial outcomes

Predict whether a trial met its primary endpoint (`label=1`). Fit a separate model for each phase.

### Prepare the text

Combine the eligibility criteria, diseases, drugs, and ICD codes.

In [ ]:
text_columns = ["criteria", "diseases_cleaned", "drugs", "icdcodes"]

def trial_text(df):
    return df[text_columns].fillna("").agg(" ".join, axis=1)

### Logistic regression baselines

Fit a bag-of-words model for each phase and report the validation metrics.

In [ ]:
val_results = outcome_val[["ID", "trial_id", "phase", "label"]].copy()
part2_submission = outcome_test[["ID"]].copy()
scores = []

for phase in ["I", "II", "III"]:
    train = outcome_train[outcome_train["phase"] == phase]
    val = outcome_val[outcome_val["phase"] == phase]
    test = outcome_test[outcome_test["phase"] == phase]

    vectorizer = CountVectorizer(binary=True, min_df=3, max_features=20000)
    X_train_phase = vectorizer.fit_transform(trial_text(train))
    X_val_phase = vectorizer.transform(trial_text(val))
    X_test_phase = vectorizer.transform(trial_text(test))

    model = LogisticRegression(max_iter=500, solver="liblinear", random_state=42)
    model.fit(X_train_phase, train["label"])

    prediction = model.predict(X_val_phase)
    probability = model.predict_proba(X_val_phase)[:, 1]
    val_results.loc[val.index, "prediction"] = prediction
    val_results.loc[val.index, "probability"] = probability
    part2_submission.loc[test.index, "label"] = model.predict_proba(X_test_phase)[:, 1]

    scores.append({
        "Phase": phase,
        "F1": f1_score(val["label"], prediction, zero_division=0),
        "ROC-AUC": roc_auc_score(val["label"], probability),
        "PR-AUC": average_precision_score(val["label"], probability),
        "Log Loss": log_loss(val["label"], probability)
    })

pd.DataFrame(scores).round(4)

### Error analysis

Use your Part 1 labels and predictions to group unsuccessful validation trials by reason, matching `trial_id` to `ID`. For each phase and reason group, report the trial count and the fraction predicted successful. Put unmatched trials in a separate “no recorded stopping reason” group, and state which labels were predicted.

In [ ]:
failed_val = val_results[val_results["label"] == 0].copy()

# Add your reason-group analysis here.

### Model improvement

Try at least one change to your outcome models. Compare the validation metrics and reason-group error rates with the baseline.

In [ ]:
# Add your revised models and comparisons here.

### Save Part 2 predictions

Save success probabilities as `ID,label`. Kaggle evaluates this file using log loss. The predictions below are from the baseline models.

In [ ]:
part2_submission.to_csv("part2_submission.csv", index=False)
part2_submission.head()